# ASR Pipeline: Video to Text with Timestamps

Uses **faster-whisper** to transcribe ultrasound teaching videos.

Pipeline: Video (.mp4) → ffmpeg → Audio (.wav) → Whisper → JSON

In [1]:
from asr_pipeline import *
import json
from pathlib import Path

In [2]:
VIDEO_PATH = "UltrasoundCrawler_KeyCode_20260323_v2/output/20260520_162816_youtube/media/case_reasoning/8V649L5Q368.mp4"
assert Path(VIDEO_PATH).exists()
print(f"Video: {VIDEO_PATH}")

Video: UltrasoundCrawler_KeyCode_20260323_v2/output/20260520_162816_youtube/media/case_reasoning/8V649L5Q368.mp4


## Step 1: Extract Audio

In [3]:
audio_path = extract_audio(VIDEO_PATH, "transcripts/audio/8V649L5Q368.wav")
print(f"Audio saved: {audio_path}")

  Extracting audio: 8V649L5Q368.mp4 → 8V649L5Q368.wav
  Audio extracted: 34.7 MB
Audio saved: transcripts/audio/8V649L5Q368.wav


## Step 2: Transcribe

In [4]:
result = transcribe_audio(audio_path, model_size="base")
print(f"Language: {result['language']} | Segments: {len(result['segments'])} | Speed: {result['speed_factor']}x")

/Users/I761836/Desktop/Semester 3/live-ultrasound-video-understanding/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


  Loading Whisper model: base (device: cpu)


  Transcribing: 8V649L5Q368.wav
  Done! 223 segments in 58.4s
  Language: en (prob: 1.00)
  Duration: 1136.9s | Speed: 19.5x realtime
Language: en | Segments: 223 | Speed: 19.46x


In [5]:
# View segments
for seg in result['segments'][:20]:
    print(f"[{seg['start']:6.1f} - {seg['end']:6.1f}] {seg['text']}")

[   0.7 -    4.5] Hi, I'm Dr. John Kugler with the Stanford 25 Ultra Sound series.
[   4.5 -    7.1] Today, we're going to be learning about lung ultrasound.
[   7.1 -   12.0] Now, lung ultrasound is really important, especially in my practice as a hospitalist.
[  12.0 -   16.0] I use lung ultrasound all the time to evaluate patients for various conditions.
[  16.2 -   22.3] Today, I want to focus on pneumothorax, pleural effusions, as well as telling the
[  22.3 -   25.3] difference between a lines and b lines, which will help us to better understand
[  25.3 -   28.0] if a patient might have a pulmonary edema or an pneumonia.
[  28.0 -   32.1] To get started, we'll start with pneumothorax.
[  32.4 -   36.0] Now, as a hospitalist, I don't diagnose a lot of pneumothoracies, right?
[  36.0 -   40.1] My colleagues down on the ED are usually checking this out more than I am,
[  40.1 -   46.2] but it can be useful, especially after performing a procedure where pneumothorax is possible.
[  4

In [6]:
# Full text
full_text = ' '.join(s['text'] for s in result['segments'])
print(f"Total length: {len(full_text)} chars")
print(f"\nFirst 500 chars:\n{full_text[:500]}")

Total length: 20940 chars

First 500 chars:
Hi, I'm Dr. John Kugler with the Stanford 25 Ultra Sound series. Today, we're going to be learning about lung ultrasound. Now, lung ultrasound is really important, especially in my practice as a hospitalist. I use lung ultrasound all the time to evaluate patients for various conditions. Today, I want to focus on pneumothorax, pleural effusions, as well as telling the difference between a lines and b lines, which will help us to better understand if a patient might have a pulmonary edema or an pn


## Step 3: Full Pipeline (one call)

In [ ]:
# Or use the full pipeline function
output = transcribe_video(VIDEO_PATH, model_size="base", output_dir="transcripts")

In [ ]:
# Check saved file
saved = json.load(open("transcripts/8V649L5Q368.json"))
print(f"Saved transcript: {saved['num_segments']} segments, {saved['duration_sec']}s")
print(f"Language: {saved['language']}")
print(f"\nFirst 3 segments:")
for s in saved['segments'][:3]:
    print(f"  [{s['start']}-{s['end']}s] {s['text']}")